# Train the neural error localizers (Colab)
Trains a **GRU from scratch** and **fine-tunes CodeBERT** on programs generated by your own grammar, then compares them with the CFG parser.
**Before you start:** Runtime > Change runtime type > **T4 GPU**. On your computer, zip your project folder *without* `.venv`
(it must contain `Backend/`, `evaluation/`, `training/`).

In [ ]:
!nvidia-smi | head -n 12

## 1. Upload and unpack the project zip

In [ ]:
import glob, os, shutil, zipfile
from google.colab import files
for z in glob.glob("/content/*.zip"): os.remove(z)        # old uploads would be picked up by mistake
shutil.rmtree("/content/proj", ignore_errors=True)
files.upload()                                            # choose your project .zip
zname = max(glob.glob("/content/*.zip"), key=os.path.getmtime)   # the newest upload, even if Colab renamed it "(1)"
zipfile.ZipFile(zname).extractall("/content/proj")
hits = glob.glob("/content/proj/**/training/run_all.py", recursive=True)
if not hits:
    print("ERROR: the zip has no training/ folder. What it contains:")
    for p in sorted(glob.glob("/content/proj/**/*.py", recursive=True))[:40]: print(" ", p)
    raise SystemExit("Re-zip your project so it includes Backend/, evaluation/ and training/.")
ROOT = hits[0][: -len("/training/run_all.py")]
open(ROOT + "/training/__init__.py", "a").close()          # make sure the package marker exists
print("project root:", ROOT, "| folders:", sorted(d for d in os.listdir(ROOT) if not d.startswith(".")))

In [ ]:
!pip -q install transformers

## 2. Save results to Google Drive (Colab erases files when the session ends)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/syntax_analyzer_models"
os.makedirs(OUT, exist_ok=True)

## 3. Train and evaluate
Takes roughly 15-30 minutes on a T4. `--zero-shot` also benchmarks plain CodeBERT, which gives you the missing zero-shot numbers.
Drop `--skip-gru` / add `--skip-bert` if you want only one model.

In [ ]:
%cd $ROOT
!python -m training.run_all --out $OUT --n-train 6000 --epochs-gru 10 --epochs-bert 2 --zero-shot

### Optional: resume
If the run stopped after training (or was too slow at the evaluation stage), the models are already in Drive. This re-runs only the evaluation:

In [ ]:
%cd $ROOT
!ls $OUT
!python -m training.run_all --out $OUT --eval-only --zero-shot

## 4. Results table (copy into your report)

In [ ]:
print(open(OUT + "/results.md").read())

## 5. Download the fine-tuned model and put it in your project as `models/codebert_localizer/`

In [ ]:
!cd $OUT && zip -qr localizer.zip codebert_localizer
files.download(OUT + "/localizer.zip")